### В этом ноутбуке покажем, как мы обрабатывали и преобразовывали данные, полученные после обогащения датасета

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv('datasets/netflix_merged.tsv', sep='\t', low_memory=False)
dataset

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,popularity,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id
0,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16 00:00:00,2010,93 min,"Comedy, Adventure, Fantasy, Animation, Family",en,...,203.893,7449.0,63.80,165000000.0,752600867.0,shrek forever after,PG,"Fantasy, Comedy, Family, Romance, Animation",NaN,t1
1,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15 00:00:00,2010,148 min,"Action, Science Fiction, Adventure",en,...,156.242,37119.0,83.69,160000000.0,839030630.0,inception,PG-13,"Scifi, Music, Thriller, Action","Action & Adventure, Sci-Fi & Fantasy, Thrillers",t2
2,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17 00:00:00,2010,NaN,"Adventure, Fantasy",en,...,121.191,19327.0,77.44,250000000.0,954305868.0,harry potter and the deathly hallows part 1,NaN,NaN,NaN,t3
3,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24 00:00:00,2010,NaN,"Animation, Family, Adventure",en,...,111.762,11638.0,76.00,260000000.0,592461732.0,tangled,NaN,NaN,NaN,t4
4,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18 00:00:00,2010,NaN,"Fantasy, Adventure, Animation, Family",en,...,110.044,13259.0,78.00,165000000.0,494879471.0,how to train your dragon,NaN,NaN,NaN,t5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41466,TV Show,Wynonna Earp,NaN,"Melanie Scrofano, Shamier Anderson, Tim Rozon,...","Canada, United States",2019-07-16 00:00:00,2018,3,"International TV Shows, TV Action & Adventure,...",NaN,...,NaN,NaN,NaN,NaN,NaN,wynonna earp,TV-14,NaN,NaN,t41467
41467,TV Show,Zig & Sharko,NaN,NaN,France,2017-12-01 00:00:00,2016,1,"Kids' TV, TV Comedies",NaN,...,NaN,NaN,NaN,NaN,NaN,zig and sharko,TV-Y7,NaN,NaN,t41468
41468,TV Show,Ben & Holly's Little Kingdom,NaN,"Preston Nyman, Sian Taylor, Ian Puleston-Davie...",United Kingdom,2020-03-15 00:00:00,2009,1,Kids' TV,NaN,...,NaN,NaN,NaN,NaN,NaN,ben and holly's little kingdom,TV-Y,NaN,NaN,t41469
41469,TV Show,Black Lightning,NaN,"Cress Williams, James Remar, Marvin 'Krondon' ...",United States,2020-03-17 00:00:00,2019,3,"TV Action & Adventure, TV Dramas, TV Sci-Fi & ...",NaN,...,NaN,NaN,NaN,NaN,NaN,black lightning,TV-14,NaN,NaN,t41470


### Для начала заметим, что у нас есть колонки, которые содержат в себе какие-то категории, и не всегда по одной на объект, их лучше вынести в отдельные таблицы, например: director, cast, country, genres...
Начнем с них

In [3]:
dataset['director'].unique()

array(['Mike Mitchell', 'Christopher Nolan', 'David Yates', ...,
       'Craig Monahan', 'Raffi Ahmad, Arie Azis', 'Alexander Nevsky'],
      shape=(15902,), dtype=object)

почти 16000 уникальных значений, давайте обработаем

In [4]:
all_directors_values = dataset['director'].to_list()
all_directors = set()
for val in all_directors_values:
    if isinstance(val, str):
        all_directors.update(list(map(str.strip, val.split(','))))
len(all_directors)

17041

Выдадим каждому режиссеру свой id

In [5]:
directors = {f'd{i}' : name for i, name in enumerate(all_directors, start=1)}
directors_inv = {v : k for k, v in directors.items()}

In [6]:
directors_and_titles = []

for _, row in dataset.iterrows():
    if pd.isna(row['director']):
        continue
    title_directors = list(map(str.strip, row['director'].split(',')))
    for title_dir in title_directors:
        directors_and_titles.append([row['title_id'], directors_inv[title_dir], title_dir])

In [7]:
directors_df = pd.DataFrame(directors_and_titles, columns=['title_id', 'director_id', 'name'])

In [8]:
directors_df

,title_id,director_id,name
0,t1,d9232,Mike Mitchell
1,t2,d10040,Christopher Nolan
2,t3,d1703,David Yates
3,t4,d1094,Byron Howard
4,t4,d8534,Nathan Greno
...,...,...,...
31893,t41454,d4602,Marc Pons
31894,t41455,d4891,Timothy Woodward Jr.
31895,t41457,d15258,Raffi Ahmad
31896,t41457,d4169,Arie Azis


Теперь у нас есть отдельная таблица для режиссеров и их работ!

Проделаем то же самое с актерами

In [9]:
all_actors = set()
for act in dataset['cast']:
    if isinstance(act, str):
        all_actors.update(list(map(str.strip, act.split(','))))
len(all_actors)

92822

In [10]:
actors = {f'a{i}' : actor for i, actor in enumerate(all_actors, 1)}
actors_inv = {v : k for k, v in actors.items()}

In [11]:
actors_and_titles = []

for _, row in dataset.iterrows():
    if pd.isna(row['cast']):
        continue
    title_cast = list(map(str.strip, row['cast'].split(',')))
    for title_act in title_cast:
        actors_and_titles.append([row['title_id'], actors_inv[title_act], title_act])

In [12]:
actors_df = pd.DataFrame(actors_and_titles, columns=['title_id', 'actor_id', 'name'])
actors_df

,title_id,actor_id,name
0,t1,a37196,Mike Myers
1,t1,a870,Eddie Murphy
2,t1,a34120,Cameron Diaz
3,t1,a83982,Antonio Banderas
4,t1,a60246,Walt Dohrn
...,...,...,...
219502,t41471,a17120,Matthias Hues
219503,t41471,a83795,Oksana Sidorenko
219504,t41471,a22639,Emmanuil Vitorgan
219505,t41471,a68440,Olga Rodionova


Теперь у нас есть отдельная таблица для актеров и их работ!

### Проделаем то же самое со странами

In [13]:
all_cntrs = set()
for val in dataset['country'].to_list():
    if isinstance(val, str):
        all_cntrs.update(list(map(str.strip, val.split(','))))
len(all_cntrs)

172

In [14]:
sorted(all_cntrs)

['',
 'Afghanistan',
 'Albania',
 'Algeria',
 'Angola',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bangladesh',
 'Belarus',
 'Belgium',
 'Bhutan',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Indian Ocean Territory',
 'Brunei Darussalam',
 'Bulgaria',
 'Burkina Faso',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cayman Islands',
 'Central African Republic',
 'Chile',
 'China',
 'Colombia',
 'Congo',
 "Cote D'Ivoire",
 'Croatia',
 'Cuba',
 'Cyprus',
 'Czech Republic',
 'Czechia',
 'Denmark',
 'Dominican Republic',
 'East Germany',
 'Ecuador',
 'Egypt',
 'Estonia',
 'Ethiopia',
 'Faeroe Islands',
 'Faroe Islands',
 'Fiji',
 'Finland',
 'France',
 'French Polynesia',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Greenland',
 'Guadaloupe',
 'Guatemala',
 'Holy See',
 'Hong Kong',
 'Hungary',
 'Iceland',
 'India',
 'Indonesia',
 'Iran',
 'Iraq',
 'Ireland',
 'Israel',
 'Italy',
 'Jamaica',
 'Japan',
 'Jordan',
 'Kazakhst

Легко заметить, что есть разные названия одних и тех же стран, и это легко устранить обычным маппингом, в отличие от признаков с именами людей, кино и тд, так как страны тески редкость, а люди - нет

Так попросим же чатгпт написать маппинг

In [15]:
COUNTRY_ALIASES = {
    "": None,
    "Unknown country": None,
    "Unknown states": None,

    "Russian Federation": "Russia",
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Czech Republic": "Czechia",
    "Turkiye": "Turkey",
    "Viet Nam": "Vietnam",
    "Syrian Arab Republic": "Syria",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Macedonia": "North Macedonia",
    "Faeroe Islands": "Faroe Islands",
    "Guadaloupe": "Guadeloupe",
    "Brunei Darussalam": "Brunei",
    "Lao People's Democratic Republic": "Laos",
    "Cote D'Ivoire": "Cote d'Ivoire",
    "Palestinian Territory": "Palestine",
    "Holy See": "Vatican City",
}

def normalize_country(country: str):
    if country is None:
        return None

    country = str(country).strip()
    return COUNTRY_ALIASES.get(country, country)

In [16]:
# проверим что нас не обманули
country_map = {country: normalize_country(country) for country in sorted(all_cntrs)}
assert len(country_map) == len(all_cntrs)
country_map

{'': None,
 'Afghanistan': 'Afghanistan',
 'Albania': 'Albania',
 'Algeria': 'Algeria',
 'Angola': 'Angola',
 'Argentina': 'Argentina',
 'Armenia': 'Armenia',
 'Aruba': 'Aruba',
 'Australia': 'Australia',
 'Austria': 'Austria',
 'Azerbaijan': 'Azerbaijan',
 'Bahamas': 'Bahamas',
 'Bangladesh': 'Bangladesh',
 'Belarus': 'Belarus',
 'Belgium': 'Belgium',
 'Bhutan': 'Bhutan',
 'Bolivia': 'Bolivia',
 'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
 'Botswana': 'Botswana',
 'Brazil': 'Brazil',
 'British Indian Ocean Territory': 'British Indian Ocean Territory',
 'Brunei Darussalam': 'Brunei',
 'Bulgaria': 'Bulgaria',
 'Burkina Faso': 'Burkina Faso',
 'Cambodia': 'Cambodia',
 'Cameroon': 'Cameroon',
 'Canada': 'Canada',
 'Cayman Islands': 'Cayman Islands',
 'Central African Republic': 'Central African Republic',
 'Chile': 'Chile',
 'China': 'China',
 'Colombia': 'Colombia',
 'Congo': 'Congo',
 "Cote D'Ivoire": "Cote d'Ivoire",
 'Croatia': 'Croatia',
 'Cuba': 'Cuba',
 'Cyprus': 'Cyprus',


In [17]:
len(set(list(country_map.values())))

159

Все правильно, итого у нас 159 уникальных стран

Так как у тайтла может быть несколько стран,мы снова заведем отдельную таблицу

Выдадим id для каждой страны и постараемся не вызвать международный скандал

In [18]:
country_to_id = {country : f'c{i}' for i, country in enumerate(set(list(country_map.values())), 1)}

In [19]:
countries_and_titles = []

for _, row in dataset.iterrows():
    if pd.isna(row['country']):
        continue
    title_countries = list(map(str.strip, row['country'].split(',')))
    mapped = [country_map[country] for country in title_countries]
    for title_c in mapped:
        if mapped:
            countries_and_titles.append([row['title_id'], country_to_id[title_c], title_c])

In [20]:
countries_df = pd.DataFrame(countries_and_titles, columns=['title_id', 'country_id', 'country_name'])
countries_df
countries_df

,title_id,country_id,country_name
0,t1,c138,United States
1,t2,c65,United Kingdom
2,t2,c138,United States
3,t3,c65,United Kingdom
4,t3,c138,United States
...,...,...,...
47662,t41468,c94,France
47663,t41469,c65,United Kingdom
47664,t41470,c138,United States
47665,t41471,c32,Russia


Теперь у нас есть таблица, по которой можно найти тайтл и страну/страны в которых он был произведен

Сделаем то же самое с жанрами

In [21]:
genres = dataset['genre'].to_list() + dataset['genres'].to_list() + dataset['listed_in'].to_list()
all_genres = set()
for g in genres:
    if not pd.isna(g):
        all_genres.update(list(map(str.lower, map(str.strip, g.split(',')))))
len(all_genres)

72

In [22]:
all_genres

{'action',
 'action & adventure',
 'adventure',
 'animation',
 'anime features',
 'anime series',
 'british tv shows',
 'children & family movies',
 'classic & cult tv',
 'classic movies',
 'comedies',
 'comedy',
 'crime',
 'crime tv shows',
 'cult movies',
 'documentaries',
 'documentary',
 'documentation',
 'docuseries',
 'drama',
 'dramas',
 'european',
 'faith & spirituality',
 'family',
 'fantasy',
 'history',
 'horror',
 'horror movies',
 'independent movies',
 'international movies',
 'international tv shows',
 'kids',
 "kids' tv",
 'korean tv shows',
 'lgbtq movies',
 'movies',
 'music',
 'music & musicals',
 'mystery',
 'news',
 'reality',
 'reality tv',
 'romance',
 'romantic movies',
 'romantic tv shows',
 'sci-fi & fantasy',
 'science & nature tv',
 'science fiction',
 'scifi',
 'soap',
 'spanish-language tv shows',
 'sport',
 'sports movies',
 'stand-up comedy',
 'stand-up comedy & talk shows',
 'talk',
 'teen tv shows',
 'thriller',
 'thrillers',
 'tv action & adventure',

Снова попросим чатгпт написать маппинг

In [23]:
AGGRESSIVE_GENRE_ALIASES = {
    # action / adventure
    "action": "action_adventure",
    "action & adventure": "action_adventure",
    "adventure": "action_adventure",
    "tv action & adventure": "action_adventure",

    # animation / family / youth
    "animation": "animation_family",
    "anime features": "animation_family",
    "anime series": "animation_family",
    "children & family movies": "animation_family",
    "family": "animation_family",
    "kids": "animation_family",
    "kids' tv": "animation_family",
    "teen tv shows": "animation_family",

    # comedy / talk
    "comedies": "comedy_talk",
    "comedy": "comedy_talk",
    "tv comedies": "comedy_talk",
    "stand-up comedy": "comedy_talk",
    "stand-up comedy & talk shows": "comedy_talk",
    "talk": "comedy_talk",

    # crime / mystery / thriller
    "crime": "crime_thriller",
    "crime tv shows": "crime_thriller",
    "mystery": "crime_thriller",
    "tv mysteries": "crime_thriller",
    "thriller": "crime_thriller",
    "thrillers": "crime_thriller",
    "tv thrillers": "crime_thriller",

    # documentary / reality / news / science / history
    "documentaries": "documentary_nonfiction",
    "documentary": "documentary_nonfiction",
    "documentation": "documentary_nonfiction",
    "docuseries": "documentary_nonfiction",
    "reality": "documentary_nonfiction",
    "reality tv": "documentary_nonfiction",
    "science & nature tv": "documentary_nonfiction",
    "history": "documentary_nonfiction",
    "news": "documentary_nonfiction",

    # drama / romance / soap / lgbtq
    "drama": "drama_romance",
    "dramas": "drama_romance",
    "tv dramas": "drama_romance",
    "romance": "drama_romance",
    "romantic movies": "drama_romance",
    "romantic tv shows": "drama_romance",
    "soap": "drama_romance",
    "lgbtq movies": "drama_romance",

    # fantasy / sci-fi / horror
    "fantasy": "fantasy_scifi_horror",
    "science fiction": "fantasy_scifi_horror",
    "scifi": "fantasy_scifi_horror",
    "sci-fi & fantasy": "fantasy_scifi_horror",
    "tv sci-fi & fantasy": "fantasy_scifi_horror",
    "horror": "fantasy_scifi_horror",
    "horror movies": "fantasy_scifi_horror",
    "tv horror": "fantasy_scifi_horror",

    # music
    "music": "music",
    "music & musicals": "music",

    # war / politics / faith
    "war": "war_politics_spirituality",
    "war & politics": "war_politics_spirituality",
    "faith & spirituality": "war_politics_spirituality",

    # international / regional
    "international movies": "international_regional",
    "international tv shows": "international_regional",
    "british tv shows": "international_regional",
    "korean tv shows": "international_regional",
    "spanish-language tv shows": "international_regional",
    "european": "international_regional",

    # sport
    "sport": "sport",
    "sports movies": "sport",

    # western
    "western": "western",

    # optional: independent можно оставить отдельно или в drama
    "independent movies": "drama_romance",

    # service / weakly informative labels
    "movies": None,
    "tv movie": None,
    "tv shows": None,
    "unknown": None,

    "classic & cult tv": "drama_romance",
    "classic movies": "drama_romance",
    "cult movies": "drama_romance",
}

def normalize_genre_aggressive(genre: str):
    if genre is None:
        return None
    genre = str(genre).strip().lower()
    return AGGRESSIVE_GENRE_ALIASES.get(genre, genre)

In [24]:
genres_map = {g : normalize_genre_aggressive(g) for g in all_genres}

In [25]:
set(list(genres_map.values()))

{None,
 'action_adventure',
 'animation_family',
 'comedy_talk',
 'crime_thriller',
 'documentary_nonfiction',
 'drama_romance',
 'fantasy_scifi_horror',
 'international_regional',
 'music',
 'sport',
 'war_politics_spirituality',
 'western'}

выдадим жанрам id

In [26]:
genres_to_id = {g : f'g{i}' for i, g in enumerate(set(list(genres_map.values())), 1)}

In [27]:
genres_and_titles = []

for _, row in dataset.iterrows():
    if pd.isna(row['genre']) and pd.isna(row['genres']) and pd.isna(row['listed_in']):
        continue

    title_genres = []
    if not pd.isna(row['genre']): title_genres.extend(list(map(str.lower, map(str.strip, row['genre'].split(',')))))
    if not pd.isna(row['genres']): title_genres.extend(list(map(str.lower, map(str.strip, row['genres'].split(',')))))
    if not pd.isna(row['listed_in']): title_genres.extend(list(map(str.lower, map(str.strip, row['listed_in'].split(',')))))

    genres_mapped = list(set([genres_map[g] for g in title_genres if g in genres_map]))
    for genre_mapped in genres_mapped:
        if mapped:
            genres_and_titles.append([row['title_id'], genres_to_id[genre_mapped], genre_mapped])

In [28]:
genres_df = pd.DataFrame(genres_and_titles, columns=['title_id', 'genre_id', 'genre_name'])
genres_df

,title_id,genre_id,genre_name
0,t1,g2,action_adventure
1,t1,g4,animation_family
2,t1,g7,fantasy_scifi_horror
3,t1,g10,drama_romance
4,t1,g13,comedy_talk
...,...,...,...
79431,t41470,g2,action_adventure
79432,t41470,g7,fantasy_scifi_horror
79433,t41470,g10,drama_romance
79434,t41471,g2,action_adventure


### Чистка

В процессе сбора данных в датасет по-любому попали дупликаты: мы собирали датасет по уникальным значениям нормализованного тайтла, года релиза и типу контента - такой подход пропускает одинаковые сущности, но в то же время позволяет собрать разные сущности, например, если у фильмов похожие названия и они вышли в один год - тогда наша цель была собрать как можно больше тайтлов, а сейчас - сделать этот набор максимально уникальным 

В процессе отчистки у нас есть приоритет: лучше оставить объект с 2 пропусками, чем тот же самый, но с 5 пропусками. Для этого посчитаем количество пропусков в каждой строке

In [29]:
dataset['na_count'] = dataset.isna().sum(axis=1)
dataset

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id,na_count
0,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16 00:00:00,2010,93 min,"Comedy, Adventure, Fantasy, Animation, Family",en,...,7449.0,63.80,165000000.0,752600867.0,shrek forever after,PG,"Fantasy, Comedy, Family, Romance, Animation",NaN,t1,1
1,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15 00:00:00,2010,148 min,"Action, Science Fiction, Adventure",en,...,37119.0,83.69,160000000.0,839030630.0,inception,PG-13,"Scifi, Music, Thriller, Action","Action & Adventure, Sci-Fi & Fantasy, Thrillers",t2,0
2,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17 00:00:00,2010,NaN,"Adventure, Fantasy",en,...,19327.0,77.44,250000000.0,954305868.0,harry potter and the deathly hallows part 1,NaN,NaN,NaN,t3,4
3,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24 00:00:00,2010,NaN,"Animation, Family, Adventure",en,...,11638.0,76.00,260000000.0,592461732.0,tangled,NaN,NaN,NaN,t4,4
4,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18 00:00:00,2010,NaN,"Fantasy, Adventure, Animation, Family",en,...,13259.0,78.00,165000000.0,494879471.0,how to train your dragon,NaN,NaN,NaN,t5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41466,TV Show,Wynonna Earp,NaN,"Melanie Scrofano, Shamier Anderson, Tim Rozon,...","Canada, United States",2019-07-16 00:00:00,2018,3,"International TV Shows, TV Action & Adventure,...",NaN,...,NaN,NaN,NaN,NaN,wynonna earp,TV-14,NaN,NaN,t41467,9
41467,TV Show,Zig & Sharko,NaN,NaN,France,2017-12-01 00:00:00,2016,1,"Kids' TV, TV Comedies",NaN,...,NaN,NaN,NaN,NaN,zig and sharko,TV-Y7,NaN,NaN,t41468,10
41468,TV Show,Ben & Holly's Little Kingdom,NaN,"Preston Nyman, Sian Taylor, Ian Puleston-Davie...",United Kingdom,2020-03-15 00:00:00,2009,1,Kids' TV,NaN,...,NaN,NaN,NaN,NaN,ben and holly's little kingdom,TV-Y,NaN,NaN,t41469,9
41469,TV Show,Black Lightning,NaN,"Cress Williams, James Remar, Marvin 'Krondon' ...",United States,2020-03-17 00:00:00,2019,3,"TV Action & Adventure, TV Dramas, TV Sci-Fi & ...",NaN,...,NaN,NaN,NaN,NaN,black lightning,TV-14,NaN,NaN,t41470,9


Для начала произведем чистку по признаку `description` - это не гарантирует нам удаление всех дупликатов по нескольким причинам: у одного и того же тайтла на нетфликсе описание может менятся со временем, у некотрых тайтлов нет описания, один и тот же тайтл мог попасть датасет с описанием и без описания, но если два описания осмысленные и одинаковые, то объекты, с большой вероятностью дубликаты

In [30]:
dataset['description'].notnull().sum()

np.int64(35440)

In [31]:
dataset['description'].nunique()

35126

In [32]:
from collections import Counter
cnt = Counter(dataset[dataset['description'].notna()]['description'].to_list())

In [33]:
duplicated_descriptions = set()
for d, count in cnt.items():
    if count > 1:
        duplicated_descriptions.add(d)

In [34]:
len(duplicated_descriptions)

291

291 описание встречается более одного раза, посмотрим на них

In [35]:
duplicated_descriptions = sorted(list(duplicated_descriptions))
duplicated_descriptions

['\r\n',
 '"Everybody Loves Raymond" creator Phil Rosenthal travels the globe to take in the local cuisine and culture of Bangkok, Lisbon, Mexico City and more.',
 '15-year-old scientist Ashley Garcia explores the great unknown of modern teendom after moving across the country to pursue a career in robotics.',
 "A 15-year-old from LA spends the summer at her mom's childhood home on an island off the coast of England, where she bonds with a mysterious horse.",
 "A 1950s housewife goes to Rio de Janeiro to meet up with her husband, only to learn he's deserted her, but decides to stay and open a bossa nova club.",
 'A Haredi family living in an ultra-Orthodox neighborhood of Jerusalem reckons with love, loss and the doldrums of daily life.',
 'A New York City grad student moonlighting as a dominatrix enlists her gay BFF from high school to be her assistant.',
 'A Turkish lieutenant and the daughter of Russian nobles fight for their love against forces of family and social expectation and 

Среди этих описаний есть, которые могут подходить к разным сущностям - удалим их

In [36]:
duplicated_descriptions.remove('\r\n')
duplicated_descriptions.remove('A narrative short film.')
duplicated_descriptions.remove('A short narrative film.')

In [37]:
dataset[dataset['description'].isin(duplicated_descriptions)].sort_values(by=['description', 'na_count'])

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id,na_count
24455,TV Show,Somebody Feed Phil,NaN,Phil Rosenthal,United States of America,2018-01-12 00:00:00,2018,1 Seasons,Documentary,en,...,36.0,73.0,NaN,NaN,somebody feed phil,TV-14,Documentation,NaN,t24456,4
37008,TV Show,Somebody Feed Phil,NaN,Philip Rosenthal,United States,2020-10-30 00:00:00,2020,4 Seasons,"Docuseries, Reality TV",NaN,...,NaN,NaN,NaN,NaN,somebody feed phil,TV-14,NaN,"Docuseries, Reality TV",t37009,8
26424,TV Show,Ashley Garcia: Genius in Love,NaN,"Paulina Chávez, Reed Horstmann, Bella Podaras,...",United States of America,2020-02-17 00:00:00,2020,1 Seasons,Comedy,en,...,38.0,68.0,NaN,NaN,ashley garcia genius in love,NaN,NaN,NaN,t26425,6
36906,TV Show,The Expanding Universe of Ashley Garcia,NaN,"Paulina Chávez, Jencarlos Canela",United States,2020-12-09 00:00:00,2020,3 Seasons,"Kids' TV, TV Comedies, Teen TV Shows",NaN,...,NaN,NaN,NaN,NaN,the expanding universe of ashley garcia,TV-PG,NaN,"Kids' TV, TV Comedies, Teen TV Shows",t36907,8
23426,TV Show,Free Rein,NaN,"Jaylen Barron, Celine Buckens, Freddy Carter, ...",United States of America,2017-06-22 00:00:00,2017,1 Seasons,"Family, Kids, Drama",en,...,203.0,79.0,NaN,NaN,free rein,TV-G,"Comedy, Drama, Family, Action, Documentation",NaN,t23427,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41411,TV Show,Izzy's Koala World,NaN,"Izzy Bee, Ali Bee, Tim Bee",Australia,2020-09-15 00:00:00,2020,1,Kids' TV,NaN,...,NaN,NaN,NaN,NaN,izzy's koala world,TV-Y,NaN,NaN,t41412,9
33541,TV Show,Pokémon Journeys: The Series,NaN,"Ikue Otani, Sarah Natochenny, Zeno Robinson, J...",Japan,2020-12-04 00:00:00,2020,1 Season,"Anime Series, Kids' TV, TV Action & Adventure",NaN,...,472.0,69.0,NaN,NaN,pokemon journeys the series,TV-Y7,"Animation, Action, Scifi, Comedy, Family, Fantasy",NaN,t33542,6
36739,TV Show,Pokémon Journeys: The Series,NaN,"Ikue Otani, Sarah Natochenny, Zeno Robinson, J...",NaN,2021-03-05 00:00:00,2021,4 Seasons,NaN,NaN,...,NaN,NaN,NaN,NaN,pokemon journeys the series,TV-Y7,NaN,"Anime Series, Kids' TV, TV Action & Adventure",t36740,10
36801,TV Show,Zig & Sharko,NaN,NaN,France,2021-02-01 00:00:00,2019,2 Seasons,NaN,NaN,...,NaN,NaN,NaN,NaN,zig and sharko,TV-Y7,NaN,"Kids' TV, TV Comedies",t36802,10


Видно дупликаты: чаще всего отличие в название (символы заменены буквами, число написано словами и тд) и в годе релиза - на разных рынках релиз может происходить в разное время

In [38]:
titles_to_drop = set(dataset[dataset['description'].isin(duplicated_descriptions)]['title_id'].to_list()) - set(dataset[dataset['description'].isin(duplicated_descriptions)].sort_values(by=['description', 'na_count']).drop_duplicates('description', keep='first')['title_id'].to_list())

In [39]:
dataset.drop(index=dataset[dataset['title_id'].isin(titles_to_drop)].index, inplace=True)
dataset

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id,na_count
0,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16 00:00:00,2010,93 min,"Comedy, Adventure, Fantasy, Animation, Family",en,...,7449.0,63.80,165000000.0,752600867.0,shrek forever after,PG,"Fantasy, Comedy, Family, Romance, Animation",NaN,t1,1
1,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15 00:00:00,2010,148 min,"Action, Science Fiction, Adventure",en,...,37119.0,83.69,160000000.0,839030630.0,inception,PG-13,"Scifi, Music, Thriller, Action","Action & Adventure, Sci-Fi & Fantasy, Thrillers",t2,0
2,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17 00:00:00,2010,NaN,"Adventure, Fantasy",en,...,19327.0,77.44,250000000.0,954305868.0,harry potter and the deathly hallows part 1,NaN,NaN,NaN,t3,4
3,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24 00:00:00,2010,NaN,"Animation, Family, Adventure",en,...,11638.0,76.00,260000000.0,592461732.0,tangled,NaN,NaN,NaN,t4,4
4,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18 00:00:00,2010,NaN,"Fantasy, Adventure, Animation, Family",en,...,13259.0,78.00,165000000.0,494879471.0,how to train your dragon,NaN,NaN,NaN,t5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41451,Movie,The Interview,Craig Monahan,"Hugo Weaving, Tony Martin, Aaron Jeffery, Paul...",Australia,2016-11-01 00:00:00,1998,101,"Cult Movies, Dramas, Thrillers",NaN,...,NaN,NaN,NaN,NaN,the interview,NR,NaN,NaN,t41452,8
41454,Movie,The Outsider,Timothy Woodward Jr.,"Jon Foo, Trace Adkins, Sean Patrick Flanery, K...",United States,2019-09-15 00:00:00,2019,86,Movies,NaN,...,NaN,NaN,NaN,NaN,the outsider,TV-MA,NaN,NaN,t41455,8
41456,Movie,The Secret,"Raffi Ahmad, Arie Azis","Nagita Slavina, Raffi Ahmad, Marshanda, Roy Ma...",NaN,2018-09-15 00:00:00,2018,98,"Horror Movies, International Movies",NaN,...,NaN,NaN,NaN,NaN,the secret,TV-14,NaN,NaN,t41457,9
41463,TV Show,Wanted,NaN,"Rebecca Gibney, Geraldine Hakewill, Stephen Pe...",Australia,2018-12-13 00:00:00,2018,3,"Crime TV Shows, International TV Shows, TV Dramas",NaN,...,NaN,NaN,NaN,NaN,wanted,TV-14,NaN,NaN,t41464,9


Удалили тайтлы с одинаковыми, непустыми описаниями

#### Теперь перейдем к более грубому методу отчистки, при котором, возможно, удалятся не только дубликаты

при анализе отсортированного по тайтлу датасета методом пристального взгляда были обнаружены дупликаты

In [40]:
dataset.sort_values(by='title_normalized')

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id,na_count
12939,Movie,"""Sr.""",Chris Smith,"Robert Downey Sr., Robert Downey Jr., Chris Sm...",United States of America,2022-09-02 00:00:00,2022,NaN,Documentary,en,...,57.0,69.00,0.0,0.0,"""sr """,NaN,NaN,NaN,t12940,4
536,Movie,#1 Cheerleader Camp,Mark Quod,"Charlene Tilton, Jay Gillespie, Harmony Blosso...",United States of America,2010-07-27 00:00:00,2010,NaN,Comedy,en,...,98.0,48.00,0.0,0.0,#1 cheerleader camp,NaN,NaN,NaN,t537,4
31687,TV Show,#1 Happy Family USA,NaN,"Ramy Youssef, Alia Shawkat, Salma Hindy, Randa...",United States of America,2025-04-17 00:00:00,2025,1 Seasons,"Comedy, Animation",en,...,0.0,0.00,NaN,NaN,#1 happy family usa,NaN,NaN,NaN,t31688,6
24568,TV Show,#5règles,NaN,Tatiana Polevoy,Canada,2018-04-17 00:00:00,2018,1 Seasons,Comedy,fr,...,0.0,0.00,NaN,NaN,#5regles,NaN,NaN,NaN,t24569,6
10037,Movie,#Alive,Cho Il,"Yoo Ah-in, Park Shin-hye, Lee Hyun-wook, Jin S...",South Korea,2020-06-24 00:00:00,2020,99 min,"Action, Horror",ko,...,1859.0,72.34,6300000.0,13416285.0,#alive,TV-MA,NaN,"Horror Movies, International Movies, Thrillers",t10038,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26963,TV Show,黄逗菌,Huang Bo,NaN,China,2021-01-22 00:00:00,2021,1 Seasons,Animation,zh,...,0.0,0.00,NaN,NaN,黄逗菌,NaN,NaN,NaN,t26964,7
19706,TV Show,黎明前的抉择,NaN,"Guo Jiaming, Lan Xi, Gong Hanlin, Zhao Da, Son...",NaN,2013-11-13 00:00:00,2013,1 Seasons,Drama,zh,...,0.0,0.00,NaN,NaN,黎明前的抉择,NaN,NaN,NaN,t19707,8
21664,TV Show,黎明破晓前,NaN,"Yu Zhen, Fan Yulin, Weiwei Wang, Wan Siwei",NaN,2015-11-21 00:00:00,2015,1 Seasons,"Drama, Action & Adventure, Documentary",zh,...,0.0,0.00,NaN,NaN,黎明破晓前,NaN,NaN,NaN,t21665,8
26870,TV Show,黑河风云,"Jun Fang, Jian Chen","Wang Ban, Shu Yaoxuan, Li Lingyu, Siqin Gaowa,...",NaN,2020-03-08 00:00:00,2020,1 Seasons,Drama,zh,...,0.0,0.00,NaN,NaN,黑河风云,NaN,NaN,NaN,t26871,7


Соберем те тайтлы, чье нормализованное название и год релиза (чтобы было не так грубо) встречается чаще одно раза

In [41]:
norm_title_and_rel_year = {}
for _, row in dataset.iterrows():
    if (row['title_normalized'], row['release_year']) in norm_title_and_rel_year:
        norm_title_and_rel_year[(row['title_normalized'], row['release_year'])].append(row['title_id'])
    else:
        norm_title_and_rel_year[(row['title_normalized'], row['release_year'])] = [row['title_id']]

In [42]:
potential_dup_title_ids = set()
for ts in list(norm_title_and_rel_year.values()):
    if len(ts) > 1:
        potential_dup_title_ids.update(ts)
len(potential_dup_title_ids)

2983

In [43]:
dataset[dataset['title_id'].isin(potential_dup_title_ids)].sort_values(by=['title_normalized', 'na_count'])

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id,na_count
38863,Movie,10 jours en or,Nicolas Brossette,"Franck Dubosc, Claude Rich, Marie Kremer, Math...",France,2017-07-01 00:00:00,2012,97 min,"Comedies, Dramas, International Movies",NaN,...,NaN,NaN,NaN,NaN,10 jours en or,TV-14,NaN,"Comedies, Dramas, International Movies",t38864,7
34367,Movie,10 Jours En Or,Nicolas Brossette,"Franck Dubosc, Alain Payen, Mathis Touré, Oliv...",France,NaN,2012,95 min,NaN,NaN,...,1003.0,60.0,NaN,NaN,10 jours en or,Others,"Comedy, Drama, European",NaN,t34368,8
20347,TV Show,100 Things to Do Before High School,NaN,"Isabela Merced, Jaheem Toombs, Owen Patrick Jo...",United States of America,2014-11-11 00:00:00,2014,1 Seasons,"Comedy, Family",en,...,38.0,76.0,NaN,NaN,100 things to do before high school,NaN,NaN,NaN,t20348,6
37804,Movie,100 Things to do Before High School,NaN,"Isabela Moner, Jaheem Toombs, Owen Joyner, Jac...",United States,2019-11-02 00:00:00,2014,44 min,Movies,NaN,...,NaN,NaN,NaN,NaN,100 things to do before high school,TV-Y,NaN,Movies,t37805,8
23461,TV Show,13 Reasons Why: Beyond the Reasons,NaN,"Jay Asher, Kate Walsh, Dylan Minnette, Katheri...",United States of America,2017-03-31 00:00:00,2017,1 Seasons,Talk,en,...,31.0,66.0,NaN,NaN,13 reasons why beyond the reasons,NaN,NaN,NaN,t23462,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35651,TV Show,Zootopia,NaN,NaN,NaN,NaN,2016,NaN,NaN,NaN,...,80.0,97.0,NaN,NaN,zootopia,PG,NaN,NaN,t35652,13
35058,Movie,Zz Top: That Little Ol' Band From Texas,Sam Dunn,"Howard Bloom, Billy Gibbons, Tim Newman, Winst...",United Kingdom of Great Britain and Northern I...,NaN,2019,91 min,NaN,NaN,...,1692.0,75.0,NaN,NaN,zz top that little ol' band from texas,Others,"Documentation, Music",NaN,t35059,8
37572,Movie,ZZ TOP: THAT LITTLE OL' BAND FROM TEXAS,Sam Dunn,NaN,"United Kingdom, Canada, United States",2020-03-01 00:00:00,2019,90 min,"Documentaries, Music & Musicals",NaN,...,NaN,NaN,NaN,NaN,zz top that little ol' band from texas,TV-MA,NaN,"Documentaries, Music & Musicals",t37573,8
38000,Movie,"¡Ay, mi madre!",Frank Ariza,"Estefanía de los Santos, Secun de la Rosa, Ter...",Spain,2019-07-19 00:00:00,2019,81 min,"Comedies, International Movies",NaN,...,NaN,NaN,NaN,NaN,¡ay mi madre,TV-MA,NaN,"Comedies, International Movies",t38001,7


Невооруженным взглядом видно дупликаты, удалим их по тому же принципу - остается тот, у которого меньше пропусков

In [44]:
titles_to_drop = set(dataset[dataset['title_id'].isin(potential_dup_title_ids)].sort_values(by=['title_normalized', 'na_count'])['title_id'].to_list()) - set(dataset[dataset['title_id'].isin(potential_dup_title_ids)].sort_values(by=['title_normalized', 'na_count']).drop_duplicates(['title_normalized', 'release_year'], keep='first')['title_id'].to_list())

In [45]:
dataset.drop(index=dataset[dataset['title_id'].isin(titles_to_drop)].index, inplace=True)

### Уникальных нормализованных тайтлов все еще достаточно - 36904 (93%). Теперь найдем те тайтлы, которые встречаются более одного раза

In [46]:
titles_norm_count = Counter(dataset['title_normalized'])

In [47]:
repeated_tts = [t for t, c in titles_norm_count.items() if c > 1]
potential_dup_title_ids = dataset[dataset['title_normalized'].isin(repeated_tts)]['title_id'].to_list()
len(potential_dup_title_ids)

5023

In [48]:
dataset[dataset['title_id'].isin(potential_dup_title_ids)].sort_values(by='title_normalized')

,type,title,director,cast,country,date_added,release_year,duration,genres,language,...,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,genre,listed_in,title_id,na_count
10909,Movie,1,Andrzej Kozlowski,"Ethan Phillips, Alice Amter, London Bridges, J...",NaN,2020-09-14 00:00:00,2020,NaN,Drama,en,...,117.0,67.05,0.0,0.0,1,NaN,NaN,NaN,t10910,5
3396,Movie,1,Paul Crowder,"Michael Fassbender, Niki Lauda, Michael Schuma...",United States of America,2013-09-30 00:00:00,2013,NaN,Documentary,en,...,346.0,65.87,0.0,0.0,1,NaN,NaN,NaN,t3397,4
24066,TV Show,100 días para enamorarse,NaN,"Franco Rizzaro, Macarena Paz, Luciano Castro, ...",Argentina,2018-05-07 00:00:00,2018,1 Seasons,Soap,es,...,2.0,55.00,NaN,NaN,100 dias para enamorarse,NaN,NaN,NaN,t24067,7
25013,TV Show,100 días para enamorarse,NaN,"María Elena Swett, Luz Valdivieso, Celine Reym...",Chile,2019-12-09 00:00:00,2019,1 Seasons,"Soap, Comedy",es,...,1.0,20.00,NaN,NaN,100 dias para enamorarse,NaN,NaN,NaN,t25014,6
39166,Movie,13 Cameras,Victor Zarcoff,"PJ McCabe, Brianne Moncrief, Sarah Baldwin, Ji...",United States,2016-08-13 00:00:00,2015,90 min,"Horror Movies, Independent Movies, Thrillers",NaN,...,NaN,NaN,NaN,NaN,13 cameras,NR,NaN,"Horror Movies, Independent Movies, Thrillers",t39167,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37004,TV Show,Zumbo's Just Desserts,NaN,"Adriano Zumbo, Rachel Khoo",Australia,2020-10-31 00:00:00,2019,1 Season,"International TV Shows, Reality TV",NaN,...,NaN,NaN,NaN,NaN,zumbo's just desserts,TV-PG,NaN,"International TV Shows, Reality TV",t37005,8
22626,TV Show,Три кота,NaN,NaN,Russia,2016-10-24 00:00:00,2016,1 Seasons,"Animation, Kids, Sci-Fi & Fantasy",ru,...,5.0,54.00,NaN,NaN,три кота,NaN,NaN,NaN,t22627,8
32438,TV Show,Три Кота,NaN,NaN,Russian Federation,NaN,2015,3 Seasons,NaN,NaN,...,95.0,68.00,NaN,NaN,три кота,TV-Y,"Scifi, Comedy, Family, Animation",NaN,t32439,10
23737,TV Show,沸腾的群山,NaN,"Song Jialun, Liang Linlin, Du Zhiguo",NaN,2017-04-24 00:00:00,2017,1 Seasons,War & Politics,zh,...,0.0,0.00,NaN,NaN,沸腾的群山,NaN,NaN,NaN,t23738,8


Методом пристального взгляда было обнаружено, что дупликаты есть, и их много, удалим, оставив только те, где меньше всего пропусков

In [49]:
titles_to_drop = set(dataset[dataset['title_id'].isin(potential_dup_title_ids)]['title_id'].to_list()) - set(dataset[dataset['title_id'].isin(potential_dup_title_ids)].sort_values(by=['na_count']).drop_duplicates(['title_normalized'], keep='first')['title_id'].to_list())

In [50]:
dataset.drop(index=dataset[dataset['title_id'].isin(titles_to_drop)].index, inplace=True)

### Перейдем к удалению данных с пропусками

удалим тайтлы, про жанр которых ничего не известно

In [51]:
dataset.drop(index=dataset[(dataset['genre'].isna()) & (dataset['listed_in'].isna()) & (dataset['genres'].isna())].index, inplace=True)

и удалим колонки с жанрами, так как эти признаки вынесены в отдельную таблицу

In [52]:
dataset.drop(labels=['genre', 'genres', 'listed_in'], axis=1, inplace=True)
dataset

,type,title,director,cast,country,date_added,release_year,duration,language,description,popularity,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,title_id,na_count
0,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16 00:00:00,2010,93 min,en,A bored and domesticated Shrek pacts with deal...,203.893,7449.0,63.80,165000000.0,752600867.0,shrek forever after,PG,t1,1
1,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15 00:00:00,2010,148 min,en,"Cobb, a skilled thief who commits corporate es...",156.242,37119.0,83.69,160000000.0,839030630.0,inception,PG-13,t2,0
2,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17 00:00:00,2010,NaN,en,"Harry, Ron and Hermione walk away from their l...",121.191,19327.0,77.44,250000000.0,954305868.0,harry potter and the deathly hallows part 1,NaN,t3,4
3,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24 00:00:00,2010,NaN,en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638.0,76.00,260000000.0,592461732.0,tangled,NaN,t4,4
4,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18 00:00:00,2010,NaN,en,As the son of a Viking leader on the cusp of m...,110.044,13259.0,78.00,165000000.0,494879471.0,how to train your dragon,NaN,t5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41373,Movie,Zinzana,Majid Al Ansari,"Ali Suliman, Saleh Bakri, Yasa, Ali Al-Jabri, ...","United Arab Emirates, Jordan",2016-03-09 00:00:00,2015,96 min,NaN,Recovering alcoholic Talal wakes up inside a s...,NaN,NaN,NaN,NaN,NaN,zinzana,TV-MA,t41374,7
41375,TV Show,Zombie Dumb,NaN,NaN,NaN,2019-07-01 00:00:00,2018,2 Seasons,NaN,"While living alone in a spooky town, a young g...",NaN,NaN,NaN,NaN,NaN,zombie dumb,TV-Y7,t41376,10
41376,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01 00:00:00,2009,88 min,NaN,Looking to survive in a world taken over by zo...,NaN,NaN,NaN,NaN,NaN,zombieland,R,t41377,7
41378,Movie,Zubaan,Mozez Singh,"Vicky Kaushal, Sarah-Jane Dias, Raaghav Chanan...",India,2019-03-02 00:00:00,2015,111 min,NaN,A scrappy but poor boy worms his way into a ty...,NaN,NaN,NaN,NaN,NaN,zubaan,TV-14,t41379,7


удалим тайтлы, для которых нет описания

In [53]:
dataset.drop(index=dataset[dataset['description'].isna()].index, inplace=True)

Теперь удалим те фильмы, для которых нет следующих признаков: `director`, `cast`, `country`, `date_added`, `language`, `user_rating_score`

In [54]:
dataset.drop(index=dataset[(dataset['type'] == 'Movie') & (dataset['director'].isna()) & (dataset['cast'].isna()) & (dataset['country'].isna()) & (dataset['date_added'].isna()) & (dataset['language'].isna()) & (dataset['user_rating_score'].isna())].index, inplace=True)
dataset

,type,title,director,cast,country,date_added,release_year,duration,language,description,popularity,user_rating_size,user_rating_score,budget,revenue,title_normalized,rating,title_id,na_count
0,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16 00:00:00,2010,93 min,en,A bored and domesticated Shrek pacts with deal...,203.893,7449.0,63.80,165000000.0,752600867.0,shrek forever after,PG,t1,1
1,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15 00:00:00,2010,148 min,en,"Cobb, a skilled thief who commits corporate es...",156.242,37119.0,83.69,160000000.0,839030630.0,inception,PG-13,t2,0
2,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17 00:00:00,2010,NaN,en,"Harry, Ron and Hermione walk away from their l...",121.191,19327.0,77.44,250000000.0,954305868.0,harry potter and the deathly hallows part 1,NaN,t3,4
3,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24 00:00:00,2010,NaN,en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638.0,76.00,260000000.0,592461732.0,tangled,NaN,t4,4
4,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18 00:00:00,2010,NaN,en,As the son of a Viking leader on the cusp of m...,110.044,13259.0,78.00,165000000.0,494879471.0,how to train your dragon,NaN,t5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41373,Movie,Zinzana,Majid Al Ansari,"Ali Suliman, Saleh Bakri, Yasa, Ali Al-Jabri, ...","United Arab Emirates, Jordan",2016-03-09 00:00:00,2015,96 min,NaN,Recovering alcoholic Talal wakes up inside a s...,NaN,NaN,NaN,NaN,NaN,zinzana,TV-MA,t41374,7
41375,TV Show,Zombie Dumb,NaN,NaN,NaN,2019-07-01 00:00:00,2018,2 Seasons,NaN,"While living alone in a spooky town, a young g...",NaN,NaN,NaN,NaN,NaN,zombie dumb,TV-Y7,t41376,10
41376,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01 00:00:00,2009,88 min,NaN,Looking to survive in a world taken over by zo...,NaN,NaN,NaN,NaN,NaN,zombieland,R,t41377,7
41378,Movie,Zubaan,Mozez Singh,"Vicky Kaushal, Sarah-Jane Dias, Raaghav Chanan...",India,2019-03-02 00:00:00,2015,111 min,NaN,A scrappy but poor boy worms his way into a ty...,NaN,NaN,NaN,NaN,NaN,zubaan,TV-14,t41379,7


Мы почистили таблицу с тайтлами, теперь надо удалить неактульные тайтлы из других таблиц

In [55]:
valid_titles_id = set(dataset['title_id'].to_list())

In [66]:
for table in [directors_df, actors_df, countries_df, genres_df, actors_df]:
    table.drop(index=table[~table['title_id'].isin(valid_titles_id)].index, inplace=True)

In [67]:
for table in [directors_df, actors_df, countries_df, genres_df]:
    assert (len(set(table['title_id'].to_list()) - valid_titles_id) == 0)

In [58]:
dataset.drop(labels=['cast', 'country', 'director', 'na_count'], axis=1, inplace=True, errors='ignore')

In [59]:
dataset['duration'] = dataset['duration'].apply(lambda x: x.split(' ')[0] if pd.notna(x) else x).astype('Int64')

Сохраним таблицы

In [72]:
dataset.to_csv('datasets/titles_cleaned.tsv', sep='\t', index=False)
directors_df.to_csv('datasets/directors_cleaned.tsv', sep='\t', index=False)
countries_df.to_csv('datasets/countries_cleaned.tsv', sep='\t', index=False)
genres_df.to_csv('datasets/genres_cleaned.tsv', sep='\t', index=False)
actors_df.to_csv('datasets/actors_cleaned.tsv', sep='\t', index=False)